# Task 3 — Qwen 14B AWQ fallback development throughput
Sequential coverage diagnostic on primary development cases from the pinned RobustBench-TC release. One case is selected per nonempty benchmark × static-file stratum (69 cases). Held-out and sensitivity-only groups are excluded.
Generation settings and model revisions follow the frozen Task 1 configuration. The diagnostic records response status, token usage, request timing, session accounting and runtime metadata; it does not estimate population-weighted accuracy or routing benefit.
A fresh Colab L4 runtime is required. Setup, dependency installation and inference execute in order. Llama inference requires authorized access through the Colab `HF_TOKEN` secret. Requested temperature 0.001 is clamped to 0.01 by the reviewed vLLM stack; both values must accompany reporting. The 1,024-token cap is retained.


In [ ]:
from pathlib import Path
import subprocess, sys, json, time, re, os
from datetime import datetime, timezone
CODE_REVISION = 'adb48d1cfea33f4c10394e75a85e2cbdbaf6fe53'
MODEL_KEY = 'qwen14b_awq'
PER_STRATUM = 1
assert re.fullmatch(r'[0-9a-f]{40}', CODE_REVISION), 'CODE_REVISION must be a full 40-character Git commit SHA'
gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True)
assert len(gpu_info.strip().splitlines()) == 1 and 'L4' in gpu_info, gpu_info
observed_gpu_start = time.perf_counter()
observed_gpu_start_utc = datetime.now(timezone.utc).isoformat()
print(gpu_info)
from google.colab import drive
drive.mount('/content/drive')
project = Path('/content/uqroute-tc-throughput')
benchmark = Path('/content/robustbench-tc-throughput')
def restore_repo(path, url, revision):
    if not path.exists():
        subprocess.run(['git', 'clone', '--no-checkout', url, str(path)], check=True)
        subprocess.run(['git', '-C', str(path), 'checkout', '--detach', revision], check=True)
    actual = subprocess.check_output(['git', '-C', str(path), 'rev-parse', 'HEAD'], text=True).strip()
    assert actual == revision, 'Existing checkout differs; use a fresh runtime'
    assert not subprocess.check_output(['git', '-C', str(path), 'status', '--porcelain'], text=True).strip(), 'Checkout is dirty'
restore_repo(project, 'https://github.com/RavindraSSK/uqroute-tc.git', CODE_REVISION)
restore_repo(benchmark, 'https://github.com/WillChow66/robustbench-tc-release.git', 'd5d03180de41eb30a6c796d04a9bfbd9dad85c1d')
freeze = json.loads((project / 'docs/config/task1_single_sample_v1.json').read_text())
model_ids = {'qwen15b': 'Qwen/Qwen2.5-1.5B-Instruct', 'llama3b': 'meta-llama/Llama-3.2-3B-Instruct', 'qwen7b': 'Qwen/Qwen2.5-7B-Instruct', 'qwen14b_awq': 'Qwen/Qwen2.5-14B-Instruct-AWQ'}
model = next(m for m in freeze['models'] if m['model_id'] == model_ids[MODEL_KEY])
output_dir = Path('/content/drive/MyDrive/UQRoute-TC/throughput') / (MODEL_KEY + '-static-strata-v1')
assert MODEL_KEY == 'qwen14b_awq', 'Model configuration must select the 14B AWQ fallback'
assert model['model_id'] == 'Qwen/Qwen2.5-14B-Instruct-AWQ'
print('MODEL TO RUN:', model['model_id'])
print('OUTPUT FOLDER:', output_dir)
env = {**os.environ, 'PYTHONPATH': str(project / 'src')}
base_command = [sys.executable, '-m', 'uqroute_tc.inference.throughput', '--benchmark-root', str(benchmark), '--project-root', str(project), '--output-dir', str(output_dir), '--model-id', model['model_id'], '--per-stratum', str(PER_STRATUM)]
preview = subprocess.check_output(base_command + ['--select-only'], text=True, env=env)
selection = json.loads(preview)
print('Selected cases:', selection['case_count'])
print(json.dumps(selection['coverage'], indent=2))
print('Development-only selection ready; no model inference yet.')


## Install the previously validated serving stack
This uses the recorded vLLM version, not the latest release. Compatibility is checked in your live runtime.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm==0.30.0', 'openai>=1.50,<3', '-e', str(project)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'], check=True)
import torch, vllm
from importlib.metadata import version
import shutil, sysconfig
vllm_executable = shutil.which('vllm')
if vllm_executable is None:
    candidate = Path(sysconfig.get_path('scripts')) / 'vllm'
    assert candidate.is_file(), 'vLLM CLI missing: inspect the installation output before proceeding'
    vllm_executable = str(candidate)
assert torch.cuda.is_available() and 'L4' in torch.cuda.get_device_name(0)
assert version('vllm') == freeze['single_sample']['server']['version']
print('Torch:', torch.__version__, 'vLLM:', version('vllm'), 'GPU:', torch.cuda.get_device_name(0))
if MODEL_KEY == 'llama3b':
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')


## Start the pinned model, run and stop the owned server
This cell rejects a server already on port 8000 because its revision cannot be verified from the model name alone. It launches the pinned revision itself. No automatic retries or warm-up requests are made; the first request is included in the timing. Saved outcomes are skipped on resume.
Do not run another copy against the same output folder. For a timeout, inspect the saved record first. An explicit `--retry-errors` run preserves old attempts, including unknown-cost interruptions.

In [ ]:
import urllib.request, signal, uuid
endpoint = 'http://127.0.0.1:8000/v1'
def served_models():
    try:
        with urllib.request.urlopen(endpoint + '/models', timeout=3) as response:
            return [m['id'] for m in json.load(response)['data']]
    except Exception:
        return None
assert served_models() is None, 'A server already exists; use a fresh runtime'
server_log = Path('/content/uqroute-throughput-server.log')
settings = freeze['single_sample']['server']
server_command = [vllm_executable, 'serve', model['model_id'], '--revision', model['model_revision'], '--tokenizer-revision', model['tokenizer_revision'], '--dtype', model['dtype'], '--max-model-len', str(settings['max_model_len']), '--gpu-memory-utilization', str(settings['gpu_memory_utilization']), '--port', '8000']
if MODEL_KEY == 'qwen14b_awq':
    server_command += ['--quantization', 'awq']
server_started = time.perf_counter()
with server_log.open('w') as log:
    server = subprocess.Popen(server_command, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
run_id = uuid.uuid4().hex
runtime = {'run_id': run_id, 'code_revision': CODE_REVISION, 'model_key': MODEL_KEY, 'gpu': gpu_info.strip(), 'observed_gpu_start_utc': observed_gpu_start_utc, 'server_command': server_command, 'torch_version': torch.__version__, 'vllm_version': version('vllm'), 'openai_version': version('openai'), 'status': 'starting'}
try:
    for _ in range(180):
        if model['model_id'] in (served_models() or []):
            break
        if server.poll() is not None:
            raise RuntimeError('Server exited: ' + '\n'.join(server_log.read_text(errors='replace').splitlines()[-30:]))
        time.sleep(5)
    else:
        raise TimeoutError('Server readiness timed out; inspect the saved log')
    runtime['startup_seconds'] = time.perf_counter() - server_started
    runtime['observed_setup_seconds'] = time.perf_counter() - observed_gpu_start
    previous_sessions = set((output_dir / 'sessions').glob('*.json'))
    result = subprocess.run(base_command, env=env)
    runtime['runner_exit_code'] = result.returncode
    runtime['status'] = 'runner_finished' if result.returncode == 0 else 'runner_failed'
    if result.returncode:
        raise RuntimeError('Runner failed; inspect the records, manifest and log before retrying')
    print('AGGREGATE ACCOUNTING (includes resume/checking time):')
    print(json.dumps(json.loads((output_dir / 'summary.json').read_text()), indent=2))
    new_sessions = set((output_dir / 'sessions').glob('*.json')) - previous_sessions
    for session_path in sorted(new_sessions):
        current_session = json.loads(session_path.read_text())
        print('CURRENT SESSION:', json.dumps(current_session, indent=2))
        if current_session.get('processed') == 0:
            print('RESUME ONLY: no new model attempts; do not use aggregate throughput as fresh inference speed.')
except BaseException as exc:
    runtime['status'] = 'failed_or_interrupted'
    runtime['error_type'] = type(exc).__name__
    raise
finally:
    if server.poll() is None:
        os.killpg(server.pid, signal.SIGTERM)
        try:
            server.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(server.pid, signal.SIGKILL)
            server.wait(timeout=30)
    runtime['observed_gpu_window_seconds_lower_bound'] = time.perf_counter() - observed_gpu_start
    runtime['observed_gpu_window_hours_lower_bound'] = runtime['observed_gpu_window_seconds_lower_bound'] / 3600
    runtime['requested_temperature'] = freeze['single_sample']['temperature']
    log_text = server_log.read_text(errors='replace')
    runtime['temperature_clamp_warning_count'] = log_text.count('We have maxed it out to 0.01.')
    runtime['effective_temperature_reported_by_server'] = 0.01 if runtime['temperature_clamp_warning_count'] else None
    runtime['note'] = 'Observed one-GPU wall window, not kernel-active time or full billed allocation. Runtime remains allocated until disconnected.'
    # Keep metadata outside the runner directory: startup failures must not create a manifest-less output folder.
    runtime_dir = output_dir.parent / (MODEL_KEY + '-runtime-metadata')
    runtime_dir.mkdir(parents=True, exist_ok=True)
    (runtime_dir / (run_id + '.json')).write_text(json.dumps(runtime, indent=2) + '\n')
    (runtime_dir / (run_id + '-server.log')).write_bytes(server_log.read_bytes())
    print('Runtime metadata:', runtime_dir)
    print('Disconnect/delete the GPU runtime now if you are finished.')


## Run artifacts
The model-specific directory under `MyDrive/UQRoute-TC/throughput/` contains `manifest.json`, `summary.json`, per-session metadata and per-case attempt history. The adjacent runtime-metadata directory contains server configuration, software versions, allocation timing and server logs.
Fresh generation and resume accounting are reported separately. Completion, token-evidence validity, parser extraction and official correctness are distinct outcomes. Truncated attempts retain their token and time costs. Threshold selection and held-out evaluation are outside this diagnostic.
